# 2. Gene selection under nested cross-validation, and a leakage check

The candidate pool of notebook 1 holds far more genes than the development set has deaths, so a survival model needs a short gene list. This notebook compares several ways of choosing that list under one protocol, measures how much a selection made outside the cross-validation inflates the score, and states which method a rule fixed in advance picks. Only the development set is used.

This is the second run; a section near the end says what changed since the first and compares the two.

## Settings

The first cell loads the libraries and locates the files. The second names every fold count, seed, threshold and grid, with its reason beside it.

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"  # folds run in parallel processes; one thread each avoids oversubscribing the cores

import hashlib, json, subprocess, sys, time
from importlib.metadata import version
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from joblib import Parallel, delayed
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw
from sksurv.util import Surv

pd.set_option("display.width", 200)  # a wide table is printed with one line per row instead of being wrapped
REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True, check=True).stdout.strip())
DEVELOPMENT_FILE = REPO / "data" / "processed" / "tcga_development.parquet"  # written by notebook 1; ignored by git
DATA_RESULTS = REPO / "results" / "01_data"
RESULTS = REPO / "results" / "02_gene_selection"  # saved runs and tables of this run; tracked by git
FIRST_RUN = RESULTS / "first_run"                 # saved runs and tables of the first run, kept for the comparison

In [ ]:
OUTER_FOLDS, OUTER_SEEDS = 5, [1, 2, 3, 4, 5]  # 5-fold cross-validation repeated 5 times: 25 held-out folds
INNER_FOLDS, INNER_SEED = 5, 0                 # tuning folds inside each training part
EXPRESSED_LOG2, EXPRESSED_SHARE = 1.0, 0.2     # keep a gene with TPM >= 1, that is log2(TPM + 1) >= 1, in at least 20% of training patients
LIST_SIZES = [50, 100, 200, 500, 1000, 2000]   # list sizes offered to the two ranking methods; the first run stopped at 500
L1_SHARES = [0.25, 0.5, 0.75]                  # elastic-net mixes between pure ridge (0) and pure LASSO (1)
PATH_LENGTH, PATH_DEPTH = 50, 0.01             # 50 penalties from the largest down to 1% of it, the library default
MAX_GENES = 2000                               # longest list a penalty path may offer, equal to the largest list size above
RIDGE_GRID = [1.0, 10.0, 100.0, 1000.0, 10000.0]  # ridge penalty of the scoring model; the first run started at 100
CLINICAL_RIDGE = 0.001                         # numerical safeguard on the clinical coefficients, negligible against the data
NEWTON_STEPS, GAIN_LIMIT = 1000, 1e-9          # a Cox fit stops when one more step would improve it by less than the limit; 1000 steps guard against an endless loop
EIGEN_FLOOR = 1e-10                            # a direction carrying less than this share of the largest one is numerically empty
DAYS_PER_YEAR = 365.25                         # average length of a year, leap years included
UNO_HORIZON = 5 * DAYS_PER_YEAR                # Uno's C is truncated at 5 years
LEAKAGE_LIST_SIZE = 100                        # list size in the leakage check and in the shuffled-outcome check
SHUFFLE_SEEDS = [1, 2, 3, 4, 5]                # shuffled-outcome check: five shuffles of the outcomes
STABLE_SHARE = 0.8                             # a gene counts as stable when at least 80% of the lists contain it
TOP_GENES = 20                                 # most often selected genes shown per method
CONFIDENCE = 0.95                              # level of every interval
CHECK_SEED = 0                                 # seed of the small synthetic examples used in the checks
N_JOBS = 4                                     # outer folds evaluated in parallel, one per physical core of the machine used
# Indicator columns of the clinical model; the most common level of each variable is the reference and gets no column.
CLINICAL_LEVELS = {"stage": ["I", "III", "IV", "unknown"], "er": ["negative", "unknown"],
                   "pr": ["negative", "unknown"], "her2": ["positive", "unknown"]}
FORMS = {"genes only": "genes_only", "genes plus clinical": "genes_plus_clinical"}  # the two forms of the scoring model and their key in saved runs
INSIDE, ON_ALL = "selection inside the training fold", "selection on all patients"  # the two ways of selecting that are contrasted

## What is compared

The table names everything that is compared and says whether it looks at patient outcomes when it picks genes. A method that uses outcomes can fit noise, which is why the selection has to be repeated inside every training part. Four rows are selection methods. Three are references: they appear in the tables and are not candidates for the choice at the end. The last row was planned and not run. This cell is longer than the others because it holds the whole table.

In [ ]:
METHODS = {
    "clinical only": {"kind": "no genes", "clinical": "clinical", "uses outcomes": "picks no genes", "role": "reference",
                      "does": "no gene: a Cox model on age, stage, ER, PR and HER2"},
    "clinical only, unknown values replaced": {
        "kind": "no genes", "clinical": "clinical, unknown replaced", "uses outcomes": "picks no genes", "role": "reference",
        "does": "as clinical only, with every unknown value replaced by the most common known level of the training part"},
    "most variable genes": {"kind": "ranking by variance", "sizes": LIST_SIZES, "uses outcomes": "no", "role": "selection method",
                            "does": "ranks genes by the variance of their expression and keeps the top of the ranking"},
    "one-gene Cox ranking": {"kind": "ranking by score test", "sizes": LIST_SIZES, "uses outcomes": "yes", "role": "selection method",
                             "does": "ranks genes by a test of each gene alone in a Cox model and keeps the top of the ranking"},
    "LASSO-Cox": {"kind": "penalised Cox", "l1_shares": [1.0], "uses outcomes": "yes", "role": "selection method",
                  "does": "fits one Cox model on all genes with a penalty that sets most coefficients to zero; the remaining genes are the list"},
    "elastic-net Cox": {"kind": "penalised Cox", "l1_shares": L1_SHARES, "uses outcomes": "yes", "role": "selection method",
                        "does": "as LASSO-Cox, with a mixed penalty that lets correlated genes enter together"},
    "all genes": {"kind": "no selection", "uses outcomes": "picks no genes", "role": "reference",
                  "does": "no selection: the scoring model on every gene that passes the expression filter"}}
NOT_RUN = {"the earlier pipeline's selector": {"role": "not run", "uses outcomes": "yes, as classes",
           "does": "LASSO on survival classes; planned for the first run, and not run because its built-in cross-validation exceeded the compute limit"}}
SELECTION_METHODS = [name for name, method in METHODS.items() if method["role"] == "selection method"]
SELECTING_KINDS = ["ranking by variance", "ranking by score test", "penalised Cox"]  # kinds whose chosen lists are kept in the saved runs


def show_table(rows, first_column):
    """Render a dictionary of rows as a Markdown table, which wraps long sentences."""
    columns = list(next(iter(rows.values())))
    lines = [f"| {first_column} | " + " | ".join(columns) + " |", "|---" * (len(columns) + 1) + "|"]
    lines += [f"| {name} | " + " | ".join(str(row[column]) for column in columns) + " |" for name, row in rows.items()]
    display(Markdown("\n".join(lines)))


show_table({name: {"what it does": row["does"], "uses patient outcomes to pick genes": row["uses outcomes"], "role": row["role"]}
            for name, row in {**METHODS, **NOT_RUN}.items()}, "name")

## Fixed before running

The protocol was written down and committed before the run. The table is rendered from the settings above, so it cannot drift from the code. Harrell's C, the main metric, is the share of comparable patient pairs that a model ranks in the right order: 0.5 is chance and 1 is a perfect ranking. Uno's C is a variant that corrects for censoring. A gene list is scored by a Cox model refit on it, the scoring model, which is defined further down. Outer and inner folds are explained in the section on nested cross-validation.

In [ ]:
show_table({
    "Data": {"fixed choice": "the development set of notebook 1 only"},
    "Outcome": {"fixed choice": "overall survival as time and event"},
    "Primary metric": {"fixed choice": "Harrell's C on the outer validation folds"},
    "Secondary metric": {"fixed choice": f"Uno's C truncated at {UNO_HORIZON / DAYS_PER_YEAR:.0f} years"},
    "Resampling": {"fixed choice": f"{OUTER_FOLDS} outer folds repeated {len(OUTER_SEEDS)} times (seeds {OUTER_SEEDS[0]} to {OUTER_SEEDS[-1]}), "
                                   f"stratified by event; all tuning in {INNER_FOLDS} inner folds of each outer training part"},
    "Fitted on training parts only": {"fixed choice": "expression filter, z-scoring, age median, most common clinical level, selection, tuning, "
                                                      "the scoring model"},
    "Scoring model": {"fixed choice": "ridge Cox refit on the listed genes, in two forms, each tuned on its own: genes only (primary) and "
                                      "genes plus clinical"},
    "Tuning": {"fixed choice": f"list size in {LIST_SIZES} for the two ranking methods; a path of {PATH_LENGTH} penalties for LASSO-Cox and "
                               f"elastic-net Cox (L1 share in {L1_SHARES}), lists of at most {MAX_GENES} genes; ridge penalty of the scoring "
                               f"model in {RIDGE_GRID}; the highest mean inner-fold Harrell's C wins"},
    "Primary comparison": {"fixed choice": f"elastic-net Cox lists score higher than LASSO-Cox lists, genes only; a null if the {CONFIDENCE:.0%} "
                                           "interval of the mean paired difference includes 0"},
    "Secondary comparisons": {"fixed choice": "genes plus clinical minus clinical only, for each of the four selection methods; a null if the "
                                              "interval includes 0"},
    "Interval": {"fixed choice": "corrected resampled t for repeated cross-validation"},
    "Choice rule": {"fixed choice": "among the four selection methods, highest mean C, genes only; methods whose interval against it includes 0 are "
                                    "tied; among tied methods prefer one that uses no outcome, then the higher Jaccard index"},
    "Leakage check": {"fixed choice": f"{ON_ALL} against {INSIDE}, for one-gene Cox ranking at {LEAKAGE_LIST_SIZE} genes, LASSO-Cox and "
                                      "elastic-net Cox"},
    "Shuffled-outcome check": {"fixed choice": f"the same contrast for one-gene Cox ranking on {len(SHUFFLE_SEEDS)} shuffles of the outcomes, "
                                               f"run first; with {INSIDE} the interval must include 0.50"},
    "After the run": {"fixed choice": "the grids are not changed again within this run, whatever the grid-edge table shows"}},
    "item")

## The development set

We load the development file and compare its checksum with the one notebook 1 recorded, so the experiment provably runs on the locked split. The data are turned into plain arrays: the expression matrix, the outcome, the age, and the level of each clinical covariate.

In [ ]:
recorded = json.loads((DATA_RESULTS / "split.json").read_text())["tcga_development"]
file_sha256 = hashlib.sha256(DEVELOPMENT_FILE.read_bytes()).hexdigest()
development = pd.read_parquet(DEVELOPMENT_FILE)
gene_ids = [column for column in development.columns if column.startswith("ENSG")]
symbols = pd.read_csv(DATA_RESULTS / "gene_pool.csv").set_index("gene_id")["symbol"]
data = {"expression": development[gene_ids].to_numpy(),
        "time": development["os_days"].to_numpy(dtype=float),
        "event": development["os_event"].to_numpy(dtype=bool),
        "age": development["age"].to_numpy(dtype=float),
        **{name: development[name].to_numpy(dtype=str) for name in CLINICAL_LEVELS}}
display(Markdown(f"Development set: {len(development)} patients, {data['event'].sum()} deaths, {len(gene_ids)} candidate genes."))

In [ ]:
# The experiment is only valid on the split that notebook 1 locked.
assert file_sha256 == recorded["file sha256"], "the development file differs from the one notebook 1 wrote"
assert len(development) == recorded["patients"] and data["event"].sum() == recorded["deaths"], "the cohort counts differ"

## Preparing a fold

Every step that learns from data is fitted on the training part of a fold and then applied to the held-out part: the expression filter, the scaling of each gene to mean 0 and variance 1 (z-scoring), the median used for a missing age, and the most common level used when unknown clinical values are replaced. Using held-out patients in any of these steps is called leakage: the evaluation would then test a model on patients that helped to build it.

In [ ]:
def clinical_columns(data, train, replace_unknown):
    """Age and one indicator column per clinical level; every statistic comes from the training patients."""
    age = np.where(np.isnan(data["age"]), np.nanmedian(data["age"][train]), data["age"])
    columns = [(age - age[train].mean()) / age[train].std()]
    for name, levels in CLINICAL_LEVELS.items():
        values = data[name]
        if replace_unknown:
            known = values[train][values[train] != "unknown"]
            values = np.where(values == "unknown", pd.Series(known).mode().iloc[0], values)
        columns += [values == level for level in levels if not (replace_unknown and level == "unknown")]
    return np.column_stack(columns).astype(float)


def prepare_fold(data, train, valid):
    """Fit the filter, the scaling and the clinical columns on the training patients and apply them to both parts."""
    log_train = data["expression"][train]
    mean, sd = log_train.mean(axis=0), log_train.std(axis=0)
    sd[sd == 0] = 1.0  # a constant gene never passes the filter; this only avoids a division by zero
    clinical = {"clinical": clinical_columns(data, train, False), "clinical, unknown replaced": clinical_columns(data, train, True)}
    return {"genes_train": (log_train - mean) / sd, "genes_valid": (data["expression"][valid] - mean) / sd,
            "expressed": np.flatnonzero((log_train >= EXPRESSED_LOG2).mean(axis=0) >= EXPRESSED_SHARE),
            "variance": log_train.var(axis=0),
            **{name: (columns[train], columns[valid]) for name, columns in clinical.items()},
            "y_train": Surv.from_arrays(data["event"][train], data["time"][train]),
            "y_valid": Surv.from_arrays(data["event"][valid], data["time"][valid])}

In [ ]:
# Leakage check: nothing fitted in prepare_fold may depend on the held-out patients, so replacing them by noise must change nothing.
rng = np.random.default_rng(CHECK_SEED)
half = len(data["time"]) // 2
train, valid = np.arange(half), np.arange(half, len(data["time"]))
noisy = {**data, "expression": data["expression"].copy(), "age": data["age"].copy(), "stage": data["stage"].copy()}
noisy["expression"][valid] = rng.normal(size=(len(valid), len(gene_ids)))
noisy["age"][valid] = rng.normal(size=len(valid))
noisy["stage"][valid] = "IV"
original, scrambled = prepare_fold(data, train, valid), prepare_fold(noisy, train, valid)
assert all(np.array_equal(original[key], scrambled[key]) for key in ["genes_train", "expressed", "variance"]), \
    "a step of prepare_fold uses the held-out patients"
assert all(np.array_equal(original[key][0], scrambled[key][0]) for key in ["clinical", "clinical, unknown replaced"]), \
    "the clinical columns of the training part depend on the held-out patients"

## The scoring model

A gene list is scored by a Cox regression refit on it, the scoring model. Cox regression is the standard model for censored survival times: every patient gets a risk score, a weighted sum of the covariates, and a higher score means a higher hazard, the momentary risk of death. The refit uses a ridge penalty, which shrinks all coefficients towards zero, and is scored on patients it has not seen. It has two forms: genes only, and genes plus clinical, where the clinical covariates join the genes, practically unpenalised. An empty list scores 0.5.

In [ ]:
def scoring_model_risk(fold, columns, ridge, form, clinical="clinical"):
    """Fit the ridge Cox model on the gene columns (and the clinical columns) and return its risk score for the held-out patients."""
    train, valid = columns
    penalty = np.full(train.shape[1], ridge)
    if form == "genes plus clinical":
        clinical_train, clinical_valid = fold[clinical]
        train, valid = np.hstack([clinical_train, train]), np.hstack([clinical_valid, valid])
        penalty = np.concatenate([np.full(clinical_train.shape[1], CLINICAL_RIDGE), penalty])
    if train.shape[1] == 0:
        return np.zeros(len(valid))  # no column at all: every patient gets the same score, which Harrell's C counts as 0.5
    return valid @ fit_ridge_cox(train, fold["y_train"]["time"], fold["y_train"]["event"], penalty)


def harrell_c(fold, risk):
    return concordance_index_censored(fold["y_valid"]["event"], fold["y_valid"]["time"], risk)[0]


def uno_c(fold, risk):
    return concordance_index_ipcw(fold["y_train"], fold["y_valid"], risk, tau=UNO_HORIZON)[0]

The library's solver for the ridge Cox model needs seconds per fit with hundreds of genes, and this run fits the scoring model for every list, ridge value, form and inner fold. We fit it ourselves. The Cox likelihood takes each death and asks what chance the model gives that patient, among all still at risk, of being the one who dies. Newton's method improves the coefficients from the slope (gradient) and curvature (Hessian) of this likelihood, halving any step that would worsen the fit. It stops when one more step would gain less than the limit set above.

In [ ]:
def fit_ridge_cox(X, time, event, penalty):
    """Coefficients of a Cox model with one ridge penalty per column (patients with tied times share a risk set)."""
    died = event.astype(float)
    outside = np.where(time[None, :] >= time[event][:, None], 0.0, -np.inf)  # one row per death: 0 for a patient still at risk

    def risk_sets(beta):
        """Per death, the chance of each patient at risk of being the one who dies; and the penalised loss of these coefficients."""
        score = X @ beta
        masked = score + outside  # minus infinity gives a patient outside the risk set the weight 0
        top = masked.max(axis=1, keepdims=True)  # taken out of the exponentials, so that a large score cannot overflow
        weight = np.exp(masked - top)
        total = weight.sum(axis=1, keepdims=True)
        return weight / total, (top + np.log(total)).sum() - died @ score + 0.5 * penalty @ beta ** 2

    beta = np.zeros(X.shape[1])
    chance, loss = risk_sets(beta)
    for _ in range(NEWTON_STEPS):
        mean, expected = chance @ X, chance.sum(axis=0)  # per death the expected covariates of the one who dies; per patient the expected deaths
        gradient = X.T @ (expected - died) + penalty * beta
        weighted = X * np.sqrt(expected)[:, None]
        hessian = weighted.T @ weighted - mean.T @ mean + np.diag(penalty)
        step = np.linalg.solve(hessian, gradient)
        if gradient @ step / 2 < GAIN_LIMIT:  # the improvement that the next step promises
            return beta - step
        new_chance, new_loss = risk_sets(beta - step)
        while new_loss > loss:
            step = step / 2
            new_chance, new_loss = risk_sets(beta - step)
        beta, chance, loss = beta - step, new_chance, new_loss
    raise RuntimeError("the Cox fit did not converge")

A list can hold more genes than there are patients. The fit then needs a matrix with one row and one column per gene, which is slow for long lists. With a ridge penalty the same model can be written on far fewer columns: the patients-by-patients matrix of gene products decides the fit, and its eigenvectors give at most one column per patient with exactly the same risk scores. The assert cell checks the fit against the library, once directly and once through this shortcut.

In [ ]:
def gene_columns(fold, genes):
    """The z-scored columns of the listed genes for both parts; more genes than patients are replaced by equivalent columns."""
    train, valid = fold["genes_train"][:, genes], fold["genes_valid"][:, genes]
    if train.shape[1] <= train.shape[0]:
        return train, valid
    values, vectors = np.linalg.eigh(train @ train.T)
    keep = values > values.max() * EIGEN_FLOOR
    return vectors[:, keep] * np.sqrt(values[keep]), (valid @ train.T) @ (vectors[:, keep] / np.sqrt(values[keep]))

In [ ]:
# The fit must agree with the Cox model of the library, with two penalty levels and tied times.
# Synthetic example: 200 patients, whole-number times so that ties occur; first 12 columns, then 300 columns, more than patients.
library_tolerance, agreement = 1e-12, 1e-6  # the library is asked for a tight fit, so that the two can be compared to six decimals
rng = np.random.default_rng(CHECK_SEED)
x, t, e = rng.normal(size=(200, 300)), rng.integers(1, 40, size=200).astype(float), rng.random(200) < 0.5
penalty = np.concatenate([np.full(4, CLINICAL_RIDGE), np.full(8, RIDGE_GRID[0])])
library = CoxPHSurvivalAnalysis(alpha=penalty, tol=library_tolerance).fit(x[:, :12], Surv.from_arrays(e, t)).coef_
assert np.allclose(fit_ridge_cox(x[:, :12], t, e, penalty), library, atol=agreement), "the fit disagrees with the library's Cox model"
short_train, short_valid = gene_columns({"genes_train": x, "genes_valid": x}, np.arange(300))
shortcut_risk = {ridge: short_valid @ fit_ridge_cox(short_train, t, e, np.full(short_train.shape[1], ridge)) for ridge in RIDGE_GRID}
direct_risk = {ridge: x @ fit_ridge_cox(x, t, e, np.full(300, ridge)) for ridge in RIDGE_GRID}
library_risk = x @ CoxPHSurvivalAnalysis(alpha=RIDGE_GRID[2], tol=library_tolerance).fit(x, Surv.from_arrays(e, t)).coef_
assert short_train.shape[1] <= 200, "the shortcut must not use more columns than patients"
assert all(np.allclose(shortcut_risk[ridge], direct_risk[ridge], atol=agreement) for ridge in RIDGE_GRID), \
    "the shortcut for long lists changes the risk scores"
assert np.allclose(shortcut_risk[RIDGE_GRID[2]], library_risk, atol=agreement), "the shortcut disagrees with the library's Cox model"

## The selection methods

The ranking by variance needs no code of its own. The one-gene Cox ranking uses a score test: for each gene alone, it asks how strongly expression and hazard go together in a one-gene Cox model. The library has no fast version of this test, so the next cell computes it for all genes at once; the cell after it checks the result against the textbook formula on a small synthetic example.

In [ ]:
def cox_score_statistics(Z, time, event):
    """Score test statistic of every column of Z in its own one-gene Cox model (tied times share one risk set)."""
    order = np.argsort(-time, kind="stable")  # longest time first: a running sum is then a sum over the patients still at risk
    Z, time, event = Z[order], time[order], event[order]
    at_risk = np.searchsorted(-time, -time, side="right")[event]  # size of the risk set at each death
    mean = np.cumsum(Z, axis=0)[at_risk - 1] / at_risk[:, None]
    mean_of_squares = np.cumsum(Z ** 2, axis=0)[at_risk - 1] / at_risk[:, None]
    observed_minus_expected = (Z[event] - mean).sum(axis=0)
    return observed_minus_expected / np.sqrt((mean_of_squares - mean ** 2).sum(axis=0))

In [ ]:
# The fast score test must agree with the textbook formula computed death by death.
# Synthetic example: 40 patients, 3 genes, whole-number times up to 14 so that ties occur, about half of the patients die.
rng = np.random.default_rng(CHECK_SEED)
z, t, e = rng.normal(size=(40, 3)), rng.integers(1, 15, size=40).astype(float), rng.random(40) < 0.5
slow = np.zeros(3)
spread = np.zeros(3)
for death in np.flatnonzero(e):
    risk_set = z[t >= t[death]]
    slow += z[death] - risk_set.mean(axis=0)
    spread += risk_set.var(axis=0)
assert np.allclose(cox_score_statistics(z, t, e), slow / np.sqrt(spread)), "the fast score test disagrees with the formula"

LASSO-Cox is a Cox model on all genes with a penalty that sets most coefficients exactly to zero; the genes left with a non-zero coefficient are the list. The elastic net mixes that penalty with a ridge penalty and lets correlated genes enter together; the L1 share says how much of the mix is the LASSO penalty. Both are fitted along a path of penalties, and each penalty yields one candidate list. The function below returns every list a method offers on one training part.

In [ ]:
def candidate_lists(method, fold, settings=None):
    """All gene lists a method offers on one training part, keyed by setting. Inner folds reuse the settings of the outer fit."""
    expressed, y = fold["expressed"], fold["y_train"]
    if method["kind"] == "fixed list":
        return {"fixed": np.asarray(method["genes"], dtype=int)}
    if method["kind"] == "no selection":
        return {"all expressed genes": expressed}
    Z = fold["genes_train"][:, expressed]
    if method["kind"].startswith("ranking"):
        by_variance = method["kind"] == "ranking by variance"
        score = fold["variance"][expressed] if by_variance else np.abs(cox_score_statistics(Z, y["time"], y["event"]))
        ranking = expressed[np.argsort(-score, kind="stable")]
        return {size: ranking[:size] for size in method["sizes"]}
    lists = {}
    for share in method["l1_shares"]:
        penalties = None if settings is None else [penalty for s, penalty in settings if s == share]
        path = CoxnetSurvivalAnalysis(l1_ratio=share, n_alphas=PATH_LENGTH, alpha_min_ratio=PATH_DEPTH, alphas=penalties).fit(Z, y)
        for column, penalty in enumerate(path.alphas_):
            genes = expressed[np.flatnonzero(path.coef_[:, column])]
            if settings is not None or len(genes) <= MAX_GENES:  # the length limit is applied once, on the outer training part
                lists[(share, float(penalty))] = genes
    return lists

## Nested cross-validation

Cross-validation holds out one fold of patients, fits on the rest and scores on the held-out fold. In nested cross-validation every choice is itself made by an inner cross-validation that only sees the training part: here the setting of the method and the ridge penalty of the scoring model, chosen separately for each of its two forms. The selection is redone on every inner training part. The held-out fold of the outer loop is used once, to score the lists chosen without it. The cell is long because tuning and scoring of a fold belong together.

In [ ]:
def tune(method, data, train, outer):
    """Inner folds of the training patients: choose the setting and the ridge penalty for each form of the scoring model."""
    lists = candidate_lists(method, outer)
    scores = {form: {(setting, ridge): [] for setting in lists for ridge in RIDGE_GRID} for form in FORMS}
    inner_folds = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=INNER_SEED).split(train, data["event"][train])
    for inner_train, inner_valid in inner_folds:
        inner = prepare_fold(data, train[inner_train], train[inner_valid])
        inner_lists = candidate_lists(method, inner, settings=list(lists))
        for setting in lists:
            columns = gene_columns(inner, inner_lists[setting])
            for ridge in RIDGE_GRID:
                for form in FORMS:
                    scores[form][(setting, ridge)].append(harrell_c(inner, scoring_model_risk(inner, columns, ridge, form)))
    # A tie goes to the first entry in grid order.
    return lists, {form: max(scores[form], key=lambda key: np.mean(scores[form][key])) for form in FORMS}


def evaluate_fold(method, data, train, valid):
    """One outer fold: tune on the training patients only, then score each form of the scoring model once on the held-out patients."""
    outer = prepare_fold(data, train, valid)
    if method["kind"] == "no genes":
        lists, chosen = {"none": np.array([], dtype=int)}, {form: ("none", 0.0) for form in FORMS}
    else:
        lists, chosen = tune(method, data, train, outer)
    lengths, record = [len(genes) for genes in lists.values()], {}
    for form, key in FORMS.items():
        setting, ridge = chosen[form]
        genes = lists[setting]
        risk = scoring_model_risk(outer, gene_columns(outer, genes), ridge, form, method.get("clinical", "clinical"))
        record |= {f"harrell_{key}": harrell_c(outer, risk), f"uno_{key}": uno_c(outer, risk), f"setting_{key}": str(setting),
                   f"ridge_{key}": ridge, f"list_length_{key}": len(genes), f"shortest_list_{key}": len(genes) == min(lengths),
                   f"longest_list_{key}": len(genes) == max(lengths), f"selected_{key}": genes}
    return record

The outer folds of a run are evaluated in parallel. Every finished run is saved under `results/02_gene_selection/` with the commit that produced it and its running time; a later execution of the notebook reads the saved file, and deleting a file makes the notebook compute it again.

In [ ]:
def run_method(method, data, seeds):
    """Evaluate a method on every outer fold of every repeat. Chosen lists are saved by Ensembl ID, for the methods that select genes."""
    folds = [(seed, index, train, valid) for seed in seeds for index, (train, valid) in
             enumerate(StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=seed).split(data["event"], data["event"]))]
    records = Parallel(n_jobs=N_JOBS)(delayed(evaluate_fold)(method, data, train, valid) for _, _, train, valid in folds)
    keeps_lists = method["kind"] in SELECTING_KINDS
    return [{"repeat": seed, "fold": index,
             **{name: [gene_ids[gene] for gene in value] if name.startswith("selected") else value
                for name, value in record.items() if keeps_lists or not name.startswith("selected")}}
            for (seed, index, _, _), record in zip(folds, records)]


def file_name(*name_parts):
    """A saved run is named after what it is, in lower case with underscores."""
    return "_".join(name_parts).lower().replace("-", "_").replace(",", "").replace(" ", "_") + ".json"


def saved(*name_parts, compute):
    """Return the saved result of a run, or compute it, save it and return it."""
    path = RESULTS / file_name(*name_parts)
    if not path.exists():
        started = time.time()
        result = compute()
        commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
        path.write_text(json.dumps({"git commit": commit, "seconds": round(time.time() - started), "result": result}, indent=1))
    return json.loads(path.read_text())["result"]

Two more pieces serve the checks below: the selection run once on all patients, and a way to cross-validate a list that is already fixed.

In [ ]:
def selected_on_all(method, data):
    """Run the whole selection, tuning included, once on all patients. Returns the list for genes only, as Ensembl IDs."""
    everyone = np.arange(len(data["time"]))
    lists, chosen = tune(method, data, everyone, prepare_fold(data, everyone, everyone))
    return [gene_ids[gene] for gene in lists[chosen["genes only"][0]]]


def fixed_list(genes):
    """A method that always offers the given list of Ensembl IDs, so that only the scoring model is tuned and scored."""
    return {"kind": "fixed list", "genes": pd.Index(gene_ids).get_indexer(genes)}

## Shuffled-outcome check

Before any real outcome is modelled we test the machinery itself. We shuffle the outcomes across patients, which destroys every true link between genes and survival, and run the one-gene Cox ranking both ways: with selection inside the training fold, and with selection on all patients before the cross-validation. A correct pipeline must score about 0.5 with selection inside the training fold. The other row shows how much score can appear from selection alone.

In [ ]:
def shuffled_outcome_check():
    rows = []
    method = {"kind": "ranking by score test", "sizes": [LEAKAGE_LIST_SIZE]}
    for seed in SHUFFLE_SEEDS:
        order = np.random.default_rng(seed).permutation(len(data["time"]))
        shuffled = {**data, "time": data["time"][order], "event": data["event"][order]}
        for arm, arm_method in [(INSIDE, method), (ON_ALL, fixed_list(selected_on_all(method, shuffled)))]:
            rows += [{"shuffle": seed, "arm": arm, **record} for record in run_method(arm_method, shuffled, [seed])]
    return rows


def t_interval(values):
    """Mean and t interval of a few independent values."""
    values = np.asarray(values, dtype=float)
    half = stats.t.ppf((1 + CONFIDENCE) / 2, len(values) - 1) * values.std(ddof=1) / np.sqrt(len(values))
    return values.mean(), values.mean() - half, values.mean() + half


def shuffled_check_table(rows):
    shuffle_means = pd.DataFrame(rows).groupby(["arm", "shuffle"])["harrell_genes_only"].mean().unstack("arm")
    return pd.DataFrame({arm: t_interval(shuffle_means[arm]) for arm in [INSIDE, ON_ALL]}, index=["mean C", "interval low", "interval high"]).T


RESULTS.mkdir(parents=True, exist_ok=True)
shuffled_table = shuffled_check_table(saved("shuffled-outcome check", compute=shuffled_outcome_check))
shuffled_table.to_csv(RESULTS / "shuffled_outcome_check_table.csv")
shuffled_table.round(3)

In [ ]:
# If selection inside the training fold finds signal in shuffled outcomes, the pipeline leaks and no result below could be trusted.
assert shuffled_table.loc[INSIDE, "interval low"] <= 0.5 <= shuffled_table.loc[INSIDE, "interval high"], \
    "selection inside the training fold scores away from 0.50 on shuffled outcomes: the code leaks"

## Method comparison

Each row of the table of what is compared is now run on the real outcomes of the development set, on the same outer folds. The table gives the mean and standard deviation of Harrell's C over the folds and the mean of Uno's C, for both forms of the scoring model, and the median length of the chosen lists. The two clinical-only rows use no gene, so they have no entry under genes only; under genes plus clinical they are the references that the other rows must beat.

In [ ]:
runs = {name: pd.DataFrame(saved("method comparison", name, compute=lambda method=method: run_method(method, data, OUTER_SEEDS)))
        for name, method in METHODS.items()}


def summarise(folds, has_genes):
    row = {}
    for form, key in FORMS.items():
        shown = has_genes or form == "genes plus clinical"
        row |= {f"{form}: Harrell's C": folds[f"harrell_{key}"].mean() if shown else np.nan,
                f"{form}: SD": folds[f"harrell_{key}"].std() if shown else np.nan,
                f"{form}: Uno's C": folds[f"uno_{key}"].mean() if shown else np.nan,
                f"{form}: genes, median": folds[f"list_length_{key}"].median()}
    return row


results_table = pd.DataFrame.from_dict({name: summarise(folds, has_genes=METHODS[name]["kind"] != "no genes")
                                        for name, folds in runs.items()}, orient="index")
results_table.to_csv(RESULTS / "method_comparison_results.csv")
results_table.round(3)

A tuned value at the edge of its grid suggests that the grid was too narrow on that side; in the first run most tuned values sat at an edge, which is why the grids are wider now. The table counts, per method and form, the outer folds in which the inner folds chose the weakest or the strongest ridge penalty on offer, and the shortest or the longest list on offer. The shortest list of a penalty path is empty, so that column then counts the folds where no gene was the best choice.

In [ ]:
def edge_counts(folds, key, one_list):
    has_genes = folds[f"list_length_{key}"] > 0  # an empty list has no ridge penalty to choose
    return {"outer folds": len(folds),
            "weakest ridge chosen": int(((folds[f"ridge_{key}"] == RIDGE_GRID[0]) & has_genes).sum()),
            "strongest ridge chosen": int(((folds[f"ridge_{key}"] == RIDGE_GRID[-1]) & has_genes).sum()),
            "shortest list chosen": "one list only" if one_list else int(folds[f"shortest_list_{key}"].sum()),
            "longest list chosen": "one list only" if one_list else int(folds[f"longest_list_{key}"].sum())}


tuned_methods = [name for name, method in METHODS.items() if method["kind"] != "no genes"]
grid_edges = pd.DataFrame.from_dict({(name, form): edge_counts(runs[name], key, one_list=METHODS[name]["kind"] == "no selection")
                                     for name in tuned_methods for form, key in FORMS.items()}, orient="index")
grid_edges.index.names = ["method", "form of the scoring model"]
grid_edges.to_csv(RESULTS / "method_comparison_grid_edges.csv")
grid_edges

The figure answers one question: how the held-out score of each method spreads over the outer folds when the scoring model uses genes only. Each dot is one fold, the box spans the middle half of the folds with the median as a line, and the dotted line is chance. Compare the spread within a method with the gaps between methods.

In [ ]:
figure, axis = plt.subplots()
axis.boxplot([runs[name]["harrell_genes_only"] for name in tuned_methods], orientation="horizontal", showfliers=False)
for position, name in enumerate(tuned_methods, start=1):
    axis.plot(runs[name]["harrell_genes_only"], np.full(len(runs[name]), position), "o", alpha=0.4)  # see-through dots show overlapping folds
axis.axvline(0.5, linestyle=":", color="grey")  # chance level
axis.set_yticks(range(1, len(tuned_methods) + 1), tuned_methods)
axis.set_xlabel("Harrell's C on the held-out fold, genes only")
axis.set_ylabel("Method")
plt.show()

Two scores from the same fold share the same patients, so methods are compared by their difference on each fold. The training parts of repeated cross-validation overlap, which makes an ordinary t interval too narrow; the corrected resampled t interval widens it to account for the overlap. The primary comparison, fixed in advance, is whether elastic-net Cox lists score higher than LASSO-Cox lists. The result counts as null if the interval includes 0.

In [ ]:
def corrected_interval(differences):
    """Mean difference over repeated cross-validation folds with the corrected resampled t interval."""
    differences = np.asarray(differences, dtype=float)
    count = len(differences)
    spread = np.sqrt((1 / count + 1 / (OUTER_FOLDS - 1)) * differences.var(ddof=1))  # 1 / (folds - 1) = test size over training size
    half = stats.t.ppf((1 + CONFIDENCE) / 2, count - 1) * spread
    return differences.mean(), differences.mean() - half, differences.mean() + half


def primary_comparison(runs):
    """Sentence on elastic-net Cox against LASSO-Cox, genes only."""
    difference, low, high = corrected_interval(runs["elastic-net Cox"]["harrell_genes_only"] - runs["LASSO-Cox"]["harrell_genes_only"])
    verdict = "includes 0, so the result is null: the elastic net is not shown to beat LASSO here" if low <= 0 <= high \
        else "excludes 0, so the two methods differ"
    return (f"elastic-net Cox minus LASSO-Cox, genes only, is {difference:+.3f} on average, {CONFIDENCE:.0%} interval "
            f"{low:+.3f} to {high:+.3f}; the interval {verdict}")


primary_sentence = primary_comparison(runs)
display(Markdown(f"Primary comparison: {primary_sentence}."))

The secondary comparisons, also fixed in advance, ask whether a selected gene list adds to the clinical covariates: for each selection method, the model with genes plus clinical is compared with the clinical-only model on the same folds. A row counts as null if its interval includes 0. The last two rows are exploratory: all genes plus clinical, and the clinical-only model with unknown values replaced, both against the clinical-only model.

In [ ]:
clinical_only = runs["clinical only"]["harrell_genes_plus_clinical"]
against_clinical = pd.DataFrame.from_dict(
    {name: dict(zip(["difference to clinical only", "interval low", "interval high"],
                    corrected_interval(runs[name]["harrell_genes_plus_clinical"] - clinical_only)))
     for name in SELECTION_METHODS + ["all genes", "clinical only, unknown values replaced"]}, orient="index")
against_clinical["interval includes 0"] = (against_clinical["interval low"] <= 0) & (against_clinical["interval high"] >= 0)
against_clinical["status"] = ["registered" if name in SELECTION_METHODS else "exploratory" for name in against_clinical.index]
against_clinical.to_csv(RESULTS / "method_comparison_against_clinical_only.csv")
against_clinical.round(3)

A selection method should also return similar lists when the patients change a little. For each method we compare its genes-only lists from all outer folds pair by pair with the Jaccard index, the size of the overlap of two lists divided by the size of their union (1 means identical lists, 0 no shared gene). We also count the stable genes, those present in at least the share of lists named `STABLE_SHARE` in the settings.

In [ ]:
def jaccard(first, second):
    union = len(set(first) | set(second))
    return len(set(first) & set(second)) / union if union else 1.0  # two empty lists are identical


def stability(lists):
    lists = list(lists)
    counts = pd.Series([gene for genes in lists for gene in genes]).value_counts()
    return {"mean pairwise Jaccard": np.mean([jaccard(a, b) for a, b in combinations(lists, 2)]),
            "stable genes": int((counts >= STABLE_SHARE * len(lists)).sum()),
            "genes, median": float(np.median([len(genes) for genes in lists])),
            "genes, range": f"{min(map(len, lists))} to {max(map(len, lists))}"}


stability_table = pd.DataFrame.from_dict({name: stability(runs[name]["selected_genes_only"]) for name in SELECTION_METHODS}, orient="index")
stability_table.to_csv(RESULTS / "method_comparison_stability.csv")
stability_table.round(3)

The table lists the genes that LASSO-Cox and elastic-net Cox select most often, each with the number of outer-fold lists that contain it. Genes are shown by symbol. It describes the lists of this comparison and is not a final gene list.

In [ ]:
def most_selected(lists):
    counts = pd.Series([symbols[gene] for genes in lists for gene in genes]).value_counts().head(TOP_GENES)
    return [f"{symbol} ({count})" for symbol, count in counts.items()]


most_selected_genes = pd.DataFrame({name: pd.Series(most_selected(runs[name]["selected_genes_only"]))
                                    for name in ["LASSO-Cox", "elastic-net Cox"]}).fillna("")
most_selected_genes.index = range(1, len(most_selected_genes) + 1)
most_selected_genes.to_csv(RESULTS / "method_comparison_most_selected_genes.csv")
most_selected_genes

The choice rule was fixed in advance and applies to the four selection methods. Take the one with the highest mean C, genes only. Every method whose corrected interval against it includes 0 is tied with it. Among tied methods prefer one that uses no outcome (only the most variable genes), then the one with the higher Jaccard index. The remaining pairwise differences are exploratory. The rule gives a proposal; adopting it is a separate decision.

In [ ]:
def choice(runs, jaccard_by_method):
    """Apply the choice rule; returns the best method, its mean C, the methods tied with it, the pick and the table of differences."""
    mean_c = {name: runs[name]["harrell_genes_only"].mean() for name in SELECTION_METHODS}
    best = max(mean_c, key=mean_c.get)
    against_best = {name: corrected_interval(runs[best]["harrell_genes_only"] - runs[name]["harrell_genes_only"])
                    for name in SELECTION_METHODS if name != best}
    tied = [best] + [name for name, (_, low, high) in against_best.items() if low <= 0 <= high]
    pick = min(tied, key=lambda name: (METHODS[name]["uses outcomes"] != "no", -jaccard_by_method[name]))
    return best, mean_c[best], tied, pick, pd.DataFrame(against_best, index=["best minus method", "interval low", "interval high"]).T


best, best_c, tied, chosen, against_best_table = choice(runs, stability_table["mean pairwise Jaccard"])
against_best_table.to_csv(RESULTS / "method_comparison_against_best.csv")
display(against_best_table.round(3))
display(Markdown(f"Highest mean C, genes only: {best} ({best_c:.3f}). Tied with it: {', '.join(tied[1:]) or 'no other method'}. "
                 f"The rule picks {chosen}."))

## Leakage check

Selection on all patients means: the selection runs once on all development patients, tuning included, and then only the scoring model is cross-validated on the same outer folds. Selection inside the training fold is the nested result above. For the one-gene Cox ranking both ways use lists of the fixed length `LEAKAGE_LIST_SIZE`. The difference between the two is the score that comes from letting the held-out patients take part in the selection. All values are for genes only.

In [ ]:
RANKING = "one-gene Cox ranking"
LEAKAGE_METHODS = {RANKING: {"kind": "ranking by score test", "sizes": [LEAKAGE_LIST_SIZE]},
                   "LASSO-Cox": METHODS["LASSO-Cox"], "elastic-net Cox": METHODS["elastic-net Cox"]}


def leakage_check_table(lists_on_all, on_all, inside):
    rows = {}
    for name in LEAKAGE_METHODS:
        difference, low, high = corrected_interval(on_all[name]["harrell_genes_only"] - inside[name]["harrell_genes_only"])
        rows[name] = {f"C, {ON_ALL}": on_all[name]["harrell_genes_only"].mean(), f"C, {INSIDE}": inside[name]["harrell_genes_only"].mean(),
                      "difference": difference, "interval low": low, "interval high": high, f"genes, {ON_ALL}": len(lists_on_all[name]),
                      f"genes, {INSIDE} (median)": inside[name]["list_length_genes_only"].median()}
    return pd.DataFrame.from_dict(rows, orient="index")


lists_on_all = {name: saved("leakage check", name, "list selected on all patients", compute=lambda method=method: selected_on_all(method, data))
                for name, method in LEAKAGE_METHODS.items()}
on_all = {name: pd.DataFrame(saved("leakage check", name, ON_ALL, compute=lambda genes=genes: run_method(fixed_list(genes), data, OUTER_SEEDS)))
          for name, genes in lists_on_all.items()}
inside = {RANKING: pd.DataFrame(saved("leakage check", RANKING, INSIDE, compute=lambda: run_method(LEAKAGE_METHODS[RANKING], data, OUTER_SEEDS))),
          "LASSO-Cox": runs["LASSO-Cox"], "elastic-net Cox": runs["elastic-net Cox"]}
leakage_table = leakage_check_table(lists_on_all, on_all, inside)
leakage_table.to_csv(RESULTS / "leakage_check_table.csv")
leakage_table.round(3)

## First run and second run

Three things changed between the runs. Notebook 1 now excludes patients with the redaction flag and men, and decides unclear HER2 results by the second test. The tuning grids are wider, because in the first run the tuned values sat at the edge of their grids. The model with genes plus clinical is tuned on its own, where the first run reused the list and penalty tuned for genes only. The table sets the main numbers of the two runs side by side; both columns are read from the saved runs.

In [ ]:
def first_run(*name_parts):
    return json.loads((FIRST_RUN / file_name(*name_parts)).read_text())["result"]


def main_numbers(runs, shuffled_table, on_all, inside):
    numbers = {f"shuffled-outcome check, {INSIDE}": shuffled_table.loc[INSIDE, "mean C"],
               "clinical only": runs["clinical only"]["harrell_genes_plus_clinical"].mean()}
    numbers |= {f"{name}, genes only": runs[name]["harrell_genes_only"].mean() for name in SELECTION_METHODS}
    numbers |= {f"{name}, genes plus clinical": runs[name]["harrell_genes_plus_clinical"].mean() for name in SELECTION_METHODS}
    numbers["elastic-net Cox minus LASSO-Cox, genes only"] = corrected_interval(
        runs["elastic-net Cox"]["harrell_genes_only"] - runs["LASSO-Cox"]["harrell_genes_only"])[0]
    numbers |= {f"leakage check, {name}: difference": (on_all[name]["harrell_genes_only"] - inside[name]["harrell_genes_only"]).mean()
                for name in LEAKAGE_METHODS}
    return numbers


first = {name: pd.DataFrame(first_run("method comparison", name)) for name in ["clinical only"] + SELECTION_METHODS}
first_on_all = {name: pd.DataFrame(first_run("leakage check", name, ON_ALL)) for name in LEAKAGE_METHODS}
first_inside = {RANKING: pd.DataFrame(first_run("leakage check", RANKING, INSIDE)), "LASSO-Cox": first["LASSO-Cox"],
                "elastic-net Cox": first["elastic-net Cox"]}
both_runs = pd.DataFrame({"first run": main_numbers(first, shuffled_check_table(first_run("shuffled-outcome check")), first_on_all, first_inside),
                          "second run": main_numbers(runs, shuffled_table, on_all, inside)})
both_runs.to_csv(RESULTS / "first_run_and_second_run.csv")
both_runs.round(3)

## The supervisor's questions

Each question raised at the meeting is a heading below. The answers are rendered from the results of this second run.

In [ ]:
gain = leakage_table["difference"]
nulls = int(against_clinical.loc[SELECTION_METHODS, "interval includes 0"].sum())
answers = {
    "Why LASSO, why the elastic net or other methods, and which one is better here":
        f"Four selection methods were compared under one protocol (section Method comparison). For the two penalised methods, {primary_sentence}. "
        f"The rule fixed in advance picks {chosen}; adopting it is a decision for the meeting. Added to the clinical covariates, "
        f"a selected gene list gives a null result for {nulls} of the {len(SELECTION_METHODS)} selection methods.",
    "Use nested cross-validation":
        f"Every tuned quantity (list size, penalty, L1 share, ridge penalty of the scoring model) is chosen in {INNER_FOLDS} inner folds "
        f"of each training part. Each of the {OUTER_FOLDS * len(OUTER_SEEDS)} outer folds is used once, for scoring only "
        "(section Nested cross-validation).",
    "Use training, validation and test sets, and investigate leakage":
        "Notebook 1 locked a test set that no notebook has opened since; this notebook works inside the development set, "
        "whose inner and outer folds play the roles of training and validation. "
        f"On shuffled outcomes the pipeline scores {shuffled_table.loc[INSIDE, 'mean C']:.3f} with {INSIDE} and "
        f"{shuffled_table.loc[ON_ALL, 'mean C']:.3f} with {ON_ALL} (section Shuffled-outcome check). On real outcomes, {ON_ALL} "
        f"changes the score by {gain[RANKING]:+.3f} for the one-gene Cox ranking, {gain['LASSO-Cox']:+.3f} for LASSO-Cox "
        f"and {gain['elastic-net Cox']:+.3f} for elastic-net Cox (section Leakage check).",
    "Why LASSO was used two times":
        "The earlier pipeline ran a least-squares LASSO on the survival class twice: first on all patients, which fixed a gene "
        "universe (`select_features_lasso` in `src/preprocessing.py` on branch `deprecated`), and again inside each fold, within "
        "that universe (`scripts/03b_sage_clinical_lasso_honest.py`). The first step saw the patients that later served as "
        "validation folds. Here selection happens once, inside the training part of each fold, starting from the full candidate "
        "pool (section Nested cross-validation)."}
display(Markdown("\n\n".join(f"### {question}\n\n{answer}" for question, answer in answers.items())))

## Run record

The last block records the git commit of this execution, the package versions, the seeds, the checksum of the input file, and for each saved run of the second run the commit that computed it and its running time.

In [ ]:
def git(*arguments):
    return subprocess.run(["git", *arguments], capture_output=True, text=True, check=True).stdout.strip()


saved_runs = {path.stem: {key: json.loads(path.read_text())[key] for key in ["git commit", "seconds"]}
              for path in sorted(RESULTS.glob("*.json")) if path.name != "run_record.json"}
run_record = {
    "git commit": git("rev-parse", "HEAD"),
    "tracked files changed since that commit": git("status", "--porcelain", "--untracked-files=no").splitlines(),
    "python": sys.version.split()[0],
    "packages": {name: version(name) for name in ["numpy", "scipy", "pandas", "pyarrow", "scikit-learn", "scikit-survival",
                                                  "joblib", "matplotlib"]},
    "seeds": {"outer folds": OUTER_SEEDS, "inner folds": INNER_SEED, "shuffles": SHUFFLE_SEEDS, "checks": CHECK_SEED},
    "input file (sha256)": {"tcga_development.parquet": file_sha256},
    "saved runs": saved_runs}
(RESULTS / "run_record.json").write_text(json.dumps(run_record, indent=1))
for key, value in run_record.items():
    if key != "saved runs":
        print(f"{key}: {json.dumps(value)}")
pd.DataFrame.from_dict(saved_runs, orient="index")